# CMI and Markov length for ring of Fibonacci plaquettes

Calculating CMI with density matrices 

\begin{equation}

CMI = I(A:C|B) = S(AB) + S(BC) - S(ABC) - S(B),

\end{equation}

where $S(Q)$ is the von Neumann entropy of the $Q$ subsystem:

\begin{equation}

S(Q) = - tr ( \rho_{Q} \log \rho_Q ),

\end{equation}

with

\begin{equation}

\rho_{Q} = tr_{\bar{Q}} (\rho).

\end{equation}

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import math
from functools import reduce
import scienceplots

In general, for a row of $k$ Fibonacci plaquettes, the ground state is given by:
\begin{equation}
    \ket{\text{g.s.}}=\frac{1}{D^k}\sum_{i=0}^k\phi^i\left(\sum \frac{1}{\phi^{a}} \frac{1}{(\sqrt{\phi})^{b}}\text{ states with $i$ applied plaquettes}  \right),
\end{equation}
where $a$ is the number of unflipped (black) inner edges and $b$ is the number of flipped (red) inner edges. The ground state is normalized by the factor $D$ (raised to the power of the total number of plaquettes), which is given by:
\begin{equation}
    D = \sqrt{1+\phi},
\end{equation}
and the factor of $\phi$ is the golden ratio:
\begin{equation}
    \phi = \frac{1+\sqrt{5}}{2}.
\end{equation}

A plaquette operator is composed out a product of four Pauli $X$ operators acting on its edges. However, because of the non-trivial fusion rule of Fibonacci anyons:
\begin{equation}
    \tau \times \tau = 1+ \tau,
\end{equation}
an edge that is shared between two plaquette operators will be in superposition of flipped and unflipped bits, with different weights specified in detail by Fibonacci fusion category.

In [2]:
# ============================================================
# Fibonacci code parameters
# ============================================================

# -------------------------
# Golden ratio
# -------------------------
phi = (1+ np.sqrt(5))/2
D = np.sqrt(1+phi)

In [ ]:
# ============================================================
# Defining function computing rho_0
# ============================================================

# -------------------------
# Tensor product helper
# -------------------------
def kron_power(v, N):
    """Compute v ⊗ v ⊗ ... ⊗ v (N times)"""
    return reduce(np.kron, [v] * N)

# -------------------------
# Basis states |0> and |1>
# -------------------------
ket_0 = np.array([1, 0])
ket_1 = np.array([0, 1])

# -------------------------
# Pauli Matrices and Identity
# -------------------------

#Pauli Matrices
Pauli_X= np.array([[0, 1],[1, 0]])
Pauli_Z= np.array([[1, 0],[0, -1]])

# identity matrix 
I = np.eye(2)

# -------------------------
# #Defining local operator
# -------------------------
def local_operator(op, i, N):
    """
    Build operator acting on i-th qubit (0-based index) in N-qubit system.
    """
    ops = [I] * N
    ops[i] = op
    return reduce(np.kron, ops)


# -------------------------
# #Defining plaquette operator
# -------------------------
def plaquette_operator(i1,i2,i3,i4,N):
    """
    A = XXXX
    """
    return local_operator(Pauli_X, i1, N) @ local_operator(Pauli_X, i2, N) @ local_operator(Pauli_X, i3, N) @ local_operator(Pauli_X, i4, N)


# -------------------------
# Functions computing the gound state and rho_0 given the system size and plaquettes' edges
# -------------------------
def ground_state(N, plaquettes):
    ket_0N = kron_power(ket_0, N)
    P = np.eye(2**N)

    for p in plaquettes:
        A = plaquette_operator(*p, N)
        P = ((np.eye(2**N) + A) / np.sqrt(2)) @ P

    return P @ ket_0N

def rho_0_function(N, plaquettes):

    ket_gs = ground_state(N, plaquettes)
    rho_0 = np.outer(ket_gs, ket_gs.conj())
    return rho_0

In [ ]:
# ============================================================
# Computing rho_0 for the system of 4 plaquettes, 12 qubits 
# ============================================================

# -------------------------
# Specifying plaquette edges
# -------------------------
plaquettes_4 = [
        (0, 1, 2, 3),
        (3, 4, 5, 6),
        (6, 7, 8, 9),
        (9, 10, 11, 0),
    ]


# -------------------------
# Computing rho_0
# -------------------------
rho_0 = rho_0_function(12,plaquettes_4)
print(rho_0)
print("Density matrix shape:", rho_0.shape)
print("Number of non-zero entries:", np.count_nonzero(rho_0))